# Background

## Unsupervised Video Anomaly Detection (VAD)

### Proxy Tasks

Since unsupervised VAD datasets contain only **normal events**, the model cannot be trained directly to distinguish normal from anomalous behavior. Instead, these methods rely on a **proxy task**: an auxiliary learning objective that forces the model to learn the distribution of normal video patterns.

The two dominant proxy tasks are:

- **Reproduction-based methods (Reconstruction in the literature):**  
  The model, typically implemented as an **Autoencoder**, receives a video frame and learns to reproduce the same frame as accurately as possible. Since it is trained only on normal data, it struggles to reproduce anomalous scenes, resulting in a high reproduction error.

- **Prediction-based methods:**  
  The model receives a sequence of previous frames and predicts the next frame. This requires learning not only the spatial appearance of the scene but also its temporal dynamics. When an abnormal event occurs, the prediction error increases because the observed motion deviates from the learned normal behavior.

Although reconstruction-based methods were introduced first, prediction-based approaches have become the dominant paradigm because they explicitly model temporal information, which is essential for detecting anomalies in videos.

---

## Limitations of Existing VAD Methods

### CNN-based Methods

CNNs are computationally efficient and scalable, but their **local receptive field** limits their ability to capture long-range spatial dependencies and global context.

### Transformer-based Methods

Transformers overcome this limitation through **self-attention**, allowing them to model global spatial relationships and temporal dynamics across video frames. However, this comes at the cost of **quadratic computational complexity**, making them difficult to deploy in real-world surveillance systems.

### Why Mamba?

Mamba offers an attractive alternative because it can model **long-range dependencies with linear complexity**, combining global context modeling with significantly lower computational cost.

---
# STNMamba

## Motivation

Although Mamba is promising, the authors argue that vanilla VMamba cannot be directly applied to Video Anomaly Detection. The authors identify **two main limitations** when directly applying vanilla Mamba to Video Anomaly Detection (VAD).

### 1. Vanilla Mamba is not optimized for spatial-temporal normality learning

#### a) Lack of Multi-scale Spatial Learning

VMamba models dependencies between image patches using multiple scanning directions, but it processes features at a **single spatial scale**. In VAD, anomalies can appear at very different sizes, from a small object (e.g., a weapon or a hand stealing an item) to a large object (e.g., a person running).

As a result, a single-scale representation may fail to capture anomalies of different sizes.

**Example**

```text
Image

+--------------------------------------+
|                                      |
|          Person running              |  <-- Large anomaly
|                                      |
|        Wallet                        |  <-- Small anomaly
|                                      |
+--------------------------------------+

Single-scale features may represent one of them well,
but not necessarily both.
```

---

#### b) Redundant Temporal Hidden States

Mamba is designed to model long-range temporal dependencies by maintaining a hidden state throughout the sequence. However, long video sequences generate a large number of hidden states, many of which contain very similar information.

This redundancy can make it harder for the model to focus on the most informative temporal changes.

**Example**

```text
Frame 100  Walking
Frame 101  Walking
Frame 102  Walking
Frame 103  Walking
Frame 104  Running  <-- Anomaly starts
Frame 105  Running

Hidden states

h100
h101
h102
h103
h104
h105
```

The first four hidden states encode nearly identical information (walking), while only one state captures the important behavioral change. The abundance of similar states may dilute the significance of the anomalous transition.

---

### 2. Current VAD Architectures Do Not Exploit Multi-level Spatial-Temporal Features

Existing VAD methods typically follow either:

- **Single-stream architectures**, where spatial and temporal information are mixed throughout the network.
- **Dual-stream architectures**, where spatial and temporal features are extracted independently and fused only at the end.

Both approaches overlook the interaction between **multiple representation levels**.

In deep neural networks, different layers capture different types of information:

- **Shallow layers:** edges, textures, local motion, fine appearance details.
- **Deep layers:** semantic concepts, long-term behaviors, and high-level context.

Current methods usually fuse spatial and temporal information only once, ignoring how these representations complement each other across different feature levels.

**Example**

```text
Spatial branch                 Temporal branch

Level 1 (edges)          Level 1 (short motion)
       │                         │
Level 2 (textures)       Level 2 (local dynamics)
       │                         │
Level 3 (objects)        Level 3 (motion patterns)
       │                         │
Level 4 (semantics)      Level 4 (long-term behavior)

Current methods:
                 ↓
          Single fusion

Proposed idea:
Fuse spatial and temporal information
at multiple levels of the network.
```

The authors argue that exploiting **multi-level spatial-temporal interactions** allows Mamba to better model complex video dynamics and improve anomaly detection performance.

---

## Proposed Solution

To address these limitations, STNMamba introduces four main components.

### 1. Multi-Scale Spatial Encoder

The Spatial Encoder incorporates **Multi-Scale Vision State Space Blocks (MS-VSSB)**.

Main idea:

- extract appearance features at multiple spatial scales;
- improve the detection of anomalies of different object sizes.

MS-VSSB extends the standard VSSB by adding parallel depth-wise convolutions with different kernel sizes before the Mamba block.

---

### 2. Channel-Aware Temporal Encoder

The Temporal Encoder employs **Channel-Aware Vision State Space Blocks (CA-VSSB)**.

Instead of Optical Flow, the model computes **RGB Differences** between consecutive frames to represent motion.

Advantages:

- much lower computational cost;
- captures short-term motion information;
- preserves important motion cues.

Additionally, CA-VSSB introduces channel attention to emphasize informative temporal features while suppressing redundant information.

---

### 3. Spatial-Temporal Interaction Module (STIM)

Unlike previous dual-stream methods that fuse appearance and motion only once, STNMamba performs **multi-level spatial-temporal interaction**.

The STIM contains:

- four Spatial-Temporal Fusion Blocks (STFB),
- four Memory Banks.

Its objective is to progressively integrate appearance and motion features throughout the network.

---

### 4. Spatial-Temporal Fusion Block (STFB)

Each STFB:

- receives spatial features;
- receives temporal features;
- projects both into a unified latent space;
- models long-range dependencies using SS2D (Mamba);
- produces a fused spatial-temporal representation.

Thus, the network learns joint appearance-motion representations at every encoder stage.

---

### 5. Memory Bank

A Memory Bank is placed after every STFB.

Its purpose is to store **spatial-temporal prototypes of normal events**.

During inference:

- query features are compared against stored prototypes;
- features close to normal prototypes are considered normal;
- features far from all prototypes contribute to higher anomaly scores.

The Memory Bank therefore enlarges the separation between normal and abnormal events in feature space.

---

## Main Results

| Dataset | STNMamba | Previous Best | Improvement |
|---------|---------:|--------------:|------------:|
| UCSD Ped2 | **98.0%** | 97.7% | **+0.3%** |
| CUHK Avenue | **89.0%** | 88.5% | **+0.5%** |
| ShanghaiTech | **74.9%** | 74.2% | **+0.7%** |

---

### Efficiency Comparison

| Method | FLOPs (G) ↓ | Params (M) ↓ | FPS ↑ |
|---------|------------:|-------------:|------:|
| MNAD | 46.6 | 15.6 | **65** |
| MemAE | 33.0 | 6.5 | 38 |
| ASTT (Transformer) | 18.9 | 63.2 | 35 |
| **STNMamba** | **1.5** | **7.2** | **40** |

#### Key Takeaways

- First Mamba-based architecture for Video Anomaly Detection.
- Learns appearance using **MS-VSSB**.
- Learns motion using **CA-VSSB** with RGB Differences instead of Optical Flow.
- Introduces **multi-level spatial-temporal fusion** through STIM and STFB.
- Uses **Memory Banks** to store prototypes of normal events.
- Achieves state-of-the-art performance while remaining lightweight and computationally efficient.